# Control the Rust IV engine from Python

For Python users who want CLI/Notebook automation. Install the thin client from a local checkout with `python -m pip install ./bindings/python` or `uv pip install ./bindings/python`, then install KohdaLab IV separately. The tutorial uses simulation only.

1. Load the thin client
2. Read live I/V without enabling output
3. Run a sweep and inspect Rust CSV
4. Verify cleanup


In [ ]:
import tempfile, csv
from pathlib import Path
from kohdalab_iv_native import NativeIV
# NativeIV() finds the Windows desktop installation; on macOS pass
# NativeIV(executable="/absolute/path/to/iv-cli") when needed.
iv_client = NativeIV


## Live samples and a sweep

The context manager waits for Rust cleanup even if the cell raises an exception. Idle samples do not create CSV rows. Values remain in SI units internally; the GUI formats display units separately.


In [ ]:
output = Path(tempfile.mkdtemp(prefix="iv-notebook-")) / "simulation.csv"
with iv_client() as iv:
    connected = iv.connect(simulation=True)
    first = iv.live()
    second = iv.live()
    assert first["timestamp"] != second["timestamp"]
    assert iv.snapshot["output"] == "Off"
    result = iv.run(output, {"pattern": "linear", "start": -0.001, "stop": 0.001, "step": 0.001, "settle_s": 0.0, "start_settle_s": 0.0})
    assert result["output"] == "Off"
    with output.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    assert len(rows) == result["points"]
    print(output, len(rows), "points, output", result["output"])
    print(rows[0])


## Exercise

Run a second simulated sweep at half the step size. Use a new filename and confirm the number of rows. Do not change `simulation=True` without checking the actual wiring and permitted limits.


In [ ]:
# Answer scaffold: change step to 0.0005 and choose a new CSV path.
# with NativeIV() as iv:
#     iv.connect(simulation=True)
#     result = iv.run(output.with_name("second.csv"), {"pattern": "linear", "step": 0.0005})


## Pitfalls and extensions

An existing output filename is refused. A GUI using the same hardware must be disconnected first. `live()` returns the next sample; `snapshot` may contain a held reading after disconnection. Source level is not an independent voltage/current measurement. For an extension, plot the simulation CSV using your preferred plotting library; no plotting dependency is required by this client.
